# HierarchicalDet Phase 0 setup (Kaggle GPU notebook)

Scope: environment install, DENTEX dataset download + 3-tier verification, and a single-image forward-pass smoke test of the inference pipeline (no pretrained HierarchicalDet weights are publicly released, so this only proves the pipeline runs mechanically -- Phase 2 covers actual training/eval).

**Before running**: enable GPU (Settings > Accelerator > GPU T4x2 or P100) and Internet (Settings > Internet > On).

This notebook expects the patched repo (with `requirements.txt`, `hierarchialdet/dataset_mapper_patched.py`, the `pycocotools/coco.py` relative-import fix) to be reachable via git. If those local changes haven't been pushed yet, either push them to the `origin` remote first, or upload the local working tree as a Kaggle Dataset and adjust the clone cell below to copy from `/kaggle/input/<dataset-name>` instead.

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU in notebook Settings before proceeding.'

In [ ]:
# Clone the repo (Phase 0 patches: requirements.txt, dataset_mapper_patched.py,
# pycocotools fix). Requires the local changes to have been pushed to origin first.
REPO_URL = 'https://github.com/christopherh-88/HierarchicalDet.git'
%cd /kaggle/working
!git clone $REPO_URL repo
%cd /kaggle/working/repo

In [ ]:
# Kaggle images ship a CUDA-matched torch already; only fill in the rest of
# requirements.txt (skip the torch/torchvision/torchaudio lines to avoid
# clobbering the preinstalled CUDA build).
!grep -v -E '^(torch|torchvision|torchaudio)$' requirements.txt > /tmp/reqs_no_torch.txt
!pip install -q -r /tmp/reqs_no_torch.txt
!pip install -q 'pillow<10'  # old detectron2 snapshot uses PIL.Image.LINEAR, removed in Pillow 10

In [ ]:
# Same fix as the local Mac setup: bundled pycocotools/ has no compiled _mask
# extension, and repo-root imports shadow the pip-installed package.
import site, glob, shutil
site_pkgs = site.getsitepackages()[0]
mask_so = glob.glob(f'{site_pkgs}/pycocotools/_mask*.so')
assert mask_so, f'no compiled _mask extension found under {site_pkgs}/pycocotools'
shutil.copy(mask_so[0], 'pycocotools/')
print('copied', mask_so[0])

In [ ]:
# Verify the full import chain, same checks as local setup.sh
import detectron2
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from hierarchialdet import add_diffusiondet_config, DiffusionDetWithTTA
from hierarchialdet.dataset_mapper_patched import DiffusionDetDatasetMapper
from pycocotools.coco import COCO
import evaluator
print('All imports OK.')

## Download DENTEX and verify all three annotation tiers

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import snapshot_download

dataset_dir = snapshot_download(
    repo_id='ibrahimhamamci/DENTEX',
    repo_type='dataset',
    local_dir='/kaggle/working/dentex_raw',
)
print(dataset_dir)

In [ ]:
# Confirm quadrant / quadrant-enumeration / quadrant-enumeration-diagnosis
# annotation files are all present, and that each is loadable as valid COCO-ish
# JSON. Adjust the glob patterns after inspecting the actual downloaded tree --
# HF dataset repo layout may not match the bundled sample filenames
# (pycocotools/custom_train.json, pycocotools/val.json) used for local smoke tests.
import glob, json, os

for root, _, files in os.walk(dataset_dir):
    for f in files:
        if f.endswith('.json'):
            print(os.path.join(root, f))

In [ ]:
# Once the three tier JSON paths are confirmed from the listing above, verify
# each loads and report image/annotation counts + category tiers present.
tier_paths = {
    'quadrant': None,                     # fill in from the listing above
    'quadrant-enumeration': None,
    'quadrant-enumeration-diagnosis': None,
}
for tier, path in tier_paths.items():
    if path is None:
        print(f'{tier}: PATH NOT SET -- fill in from the file listing above')
        continue
    d = json.load(open(path))
    print(tier, path)
    print('  images:', len(d.get('images', [])), 'annotations:', len(d.get('annotations', [])))
    print('  category keys:', [k for k in d if k.startswith('categor')])

## Single-image forward-pass smoke test

No pretrained HierarchicalDet weights exist publicly, so this uses randomly
initialized weights (backbone init from the public Swin-B ImageNet-22k
checkpoint per `configs/diffdet.custom.swinbase.nonpretrain.yaml`). The goal
is only to prove the model builds and runs a forward pass on GPU without
crashing -- not to produce meaningful detections.

In [ ]:
# Download the public Swin-B ImageNet-22k backbone referenced by the nonpretrain config.
# IMPORTANT: keep the .pth extension. detectron2's DetectionCheckpointer dispatches
# purely on file extension (detectron2/checkpoint/detection_checkpoint.py:_load_file):
# ".pkl" is parsed as a Caffe2-style pickle blob, which would silently mis-load (or
# crash on) a native torch.save() file. Only a non-".pkl"/".pyth" extension routes
# through the normal torch.load() + name-matching-heuristics path, which is what a
# raw Swin-Transformer release checkpoint needs.
import os
os.makedirs('models', exist_ok=True)
!wget -q -O models/swin_base_patch4_window7_224_22k.pth \
  https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_base_patch4_window7_224_22k.pth
# NOTE: even with the extension fixed, detectron2 loads this via name-matching
# heuristics (align_and_update_state_dicts), which only works if this checkpoint's
# state_dict keys line up with hierarchialdet/swintransformer.py's module names.
# Not yet confirmed -- check the printed "Some model parameters/buffers are not
# found" warnings when demo.py runs; if most/all keys mismatch, the backbone is
# effectively randomly initialized and a key-renaming step will be needed.

In [ ]:
# Pick one real image from the downloaded dataset for the smoke test once the
# tier paths above are filled in.
!python demo.py \
  --config-file configs/diffdet.custom.swinbase.nonpretrain.yaml \
  --input <path-to-one-dentex-image> \
  --output /kaggle/working/smoke_test_output.jpg \
  --opts MODEL.WEIGHTS models/swin_base_patch4_window7_224_22k.pth MODEL.DEVICE cuda